# Calculate unit-wise difference between the Taunton-Worcester scenario and base

Aggregate differences by stroke unit.

## Code setup

In [1]:
import os
import numpy as np
import pandas as pd

from classes.calculations import calculate_extra_bits

In [2]:
dir_data = 'data'
dir_output = 'output'
# Scenario results:
file_comparison_unit_combo = 'comparison_by_unit_combo_base_taunworc'
# Reference:
file_hospitals = 'stroke_hospitals_2025.csv'
file_unit_combos = 'unit_combos.csv'
# Output:
file_comparison_nearest_unit = 'comparison_by_nearest_unit_base_taunworc'
file_comparison_transfer_unit = 'comparison_by_transfer_unit_base_taunworc'

In [3]:
strings_tl = ['', '_transferloss0!67']  # column names for transfer rates
scenarios = ['base', 'taunworc']

## Set up column names

Gather columns present in the main results dataframes.

Gather columns by type and without tailing scenario names (`_base`, `_taunworc`):

In [4]:
col_admissions = 'Admissions_2025_Est'
cols_admissions_weighted_outcomes = [
    'admissions_weighted_drip_ship_mrs_0-2',
    'admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2',
]

# Add columns affected by lossy transfer rate:
def add_str_to_columns(cols, s_list):
    return [f'{c}{s}' for c in cols for s in s_list]
    
cols_admissions_weighted_outcomes = add_str_to_columns(
    cols_admissions_weighted_outcomes, strings_tl)

## Function to combine units

Admissions can just be summed.

Admission-weighted outcomes need to be weighted more carefully.

In [5]:
def combine_units_admissions_weighted_outcomes(
        df,  # by unit, units may appear more than once
        cols_admissions_weighted_outcomes,
        col_admissions,
        col_unit
    ):
    df = df.copy()
    df[cols_admissions_weighted_outcomes] = (
        df[cols_admissions_weighted_outcomes]
            .multiply(df[col_admissions], axis='rows')
    )
    cols = [col_unit, col_admissions] + cols_admissions_weighted_outcomes
    df_here = df[cols].groupby(col_unit).sum()
    # Divide outcomes by the direct admissions to each unit:
    df_here[cols_admissions_weighted_outcomes] = (
        df_here[cols_admissions_weighted_outcomes]
            .div(df_here[col_admissions], axis='rows')
    )
    df_here = df_here.drop(col_admissions, axis='columns')
    df_here = df_here.round(5)

    return df_here

## Run calculations

This cell prints the columns that are created by each function in calculate_results_per_scenario().

In [7]:
for s_bias in ['_bias10min', '']:
    # Load in the LSOA-level outcomes and admissions data:
    p = os.path.join(dir_output, file_comparison_unit_combo + s_bias + '.csv')
    comparison_by_unit_combo = pd.read_csv(p, index_col=0)

    cols_unit = ['nearest_ivt_unit', 'transfer_unit']
    cols_to_sum = []
    
    dict_by_unit = {}
    for s in ['base', 'taunworc']:
        # Set up columns:
        cols_unit_here = [f'{c}_{s}' for c in cols_unit]
        cols_scen = [c for c in comparison_by_unit_combo.columns if s in c]
        cols_adweight = [c for c in cols_scen if 'weighted' in c]
        cols_ad = [c for c in cols_scen if
                       ((('dmissions' in c) | ('transfers_mt' in c)) &
                        (c not in cols_adweight))]
        col_ad = f'{col_admissions}_{s}'
        # Remaining columns should be nearest_ivt_unit and transfer_unit
        # and their SSNAP names and hospital cities.
        cols_left = list(set(cols_scen) - (set(cols_adweight) | set(cols_to_sum)))
    
        for col_unit in cols_unit_here:
            # Stroke unit info:
            if 'transfer' in col_unit:
                cols_this_unit = [f'SSNAP name_transfer_unit_{s}',
                                  f'Hospital_City_transfer_unit_{s}']
            else:
                cols_this_unit = [f'SSNAP name_{s}', f'Hospital_City_{s}']
            df_units = (comparison_by_unit_combo[[col_unit] + cols_this_unit]
                .drop_duplicates(ignore_index=True))
            df_units = df_units.set_index(col_unit)
    
            df_ad = comparison_by_unit_combo[[col_unit] + cols_ad].copy()
            df_ad = df_ad.groupby(col_unit).sum()
    
            df_adweight = comparison_by_unit_combo[
                [col_unit, col_ad] + cols_adweight].copy()
            df_adweight = combine_units_admissions_weighted_outcomes(
                df_adweight,  # by unit, units may appear more than once
                cols_adweight,
                col_ad,
                col_unit
            )
    
            df_here = pd.concat((df_units, df_ad, df_adweight), axis='columns')
    
            dict_by_unit[col_unit] = df_here.copy()
    
    # Combine scenarios.    
    # Keep the nearest IVT unit and transfer unit data separate
    # because they're not always the same pair between scenarios.
    comparison_by_nearest_ivt_unit = pd.concat(
        (dict_by_unit['nearest_ivt_unit_base'],
         dict_by_unit['nearest_ivt_unit_taunworc']),
        axis='columns'
    )
    comparison_by_transfer_unit = pd.concat(
        (dict_by_unit['transfer_unit_base'],
         dict_by_unit['transfer_unit_taunworc']),
        axis='columns'
    )
    
    comparison_by_nearest_ivt_unit = calculate_extra_bits(
        comparison_by_nearest_ivt_unit, scenarios)
    comparison_by_transfer_unit = calculate_extra_bits(
        comparison_by_transfer_unit, scenarios)
    
    p = os.path.join(dir_output, file_comparison_nearest_unit + s_bias + '.csv')
    comparison_by_nearest_ivt_unit.to_csv(p)
    
    p = os.path.join(dir_output, file_comparison_transfer_unit + s_bias + '.csv')
    comparison_by_transfer_unit.to_csv(p)